Load the required tables

In [0]:
from pyspark.sql import functions as F

bronze_df = spark.table(
    "customer360_dev.bronze.addresses"
)

silver_df = spark.table(
    "customer360_dev.silver.addresses"
)

reject_df = spark.table(
    "customer360_dev.quarantine.address_rejects"
)

In [0]:
bronze_count = bronze_df.count()
silver_count = silver_df.count()
reject_count = reject_df.count()

print(f"Bronze rows   : {bronze_count:,}")
print(f"Silver rows   : {silver_count:,}")
print(f"Rejected rows : {reject_count:,}")

Bronze rows   : 1,200,078
Silver rows   : 1,178,400
Rejected rows : 21,678


Rebuild valid candidate logic

In [0]:
customers_df = spark.table(
    "customer360_dev.silver.customers"
).select("customer_id")

In [0]:
validated_df = (
    bronze_df
    .withColumn(
        "effective_from_cast",
        F.to_timestamp("effective_from")
    )
    .withColumn(
        "effective_to_cast",
        F.to_timestamp("effective_to")
    )
    .withColumn(
        "address_type_clean",
        F.upper(F.trim("address_type"))
    )
    .join(
        customers_df.withColumn(
            "_customer_exists",
            F.lit(True)
        ),
        on="customer_id",
        how="left"
    )
    .withColumn(
        "dq_missing_address_id",
        F.col("address_id").isNull()
        | (F.trim(F.col("address_id")) == "")
    )
    .withColumn(
        "dq_missing_customer_id",
        F.col("customer_id").isNull()
        | (F.trim(F.col("customer_id")) == "")
    )
    .withColumn(
        "dq_customer_not_found",
        F.col("_customer_exists").isNull()
    )
    .withColumn(
        "dq_invalid_address_type",
        ~F.col("address_type_clean").isin(
            "HOME",
            "BILLING",
            "SHIPPING"
        )
    )
    .withColumn(
        "dq_invalid_effective_dates",
        F.col("effective_from_cast").isNull()
        | (
            F.col("effective_to_cast").isNotNull()
            & (
                F.col("effective_to_cast")
                < F.col("effective_from_cast")
            )
        )
    )
    .withColumn(
        "dq_critical_failure",
        F.col("dq_missing_address_id")
        | F.col("dq_missing_customer_id")
        | F.col("dq_customer_not_found")
        | F.col("dq_invalid_address_type")
        | F.col("dq_invalid_effective_dates")
    )
)

Count valid candidates before deduplication

In [0]:
valid_candidate_count = (
    validated_df
    .filter(~F.col("dq_critical_failure"))
    .count()
)

print(
    f"Valid candidates before dedup: "
    f"{valid_candidate_count:,}"
)

Valid candidates before dedup: 1,178,400


Calculate duplicates removed

In [0]:
duplicate_rows_removed = (
    valid_candidate_count - silver_count
)

print(
    f"Duplicate rows removed: "
    f"{duplicate_rows_removed:,}"
)

Duplicate rows removed: 0


Reconcile the totals

In [0]:
accounted_for = (
    silver_count
    + reject_count
    + duplicate_rows_removed
)

difference = bronze_count - accounted_for

print(f"Bronze rows        : {bronze_count:,}")
print(f"Silver rows        : {silver_count:,}")
print(f"Rejected rows      : {reject_count:,}")
print(f"Duplicates removed : {duplicate_rows_removed:,}")
print(f"Accounted for      : {accounted_for:,}")
print(f"Difference         : {difference:,}")

Bronze rows        : 1,200,078
Silver rows        : 1,178,400
Rejected rows      : 21,678
Duplicates removed : 0
Accounted for      : 1,200,078
Difference         : 0


Create reconciliation result

In [0]:
result = [
    {
        "entity": "addresses",
        "bronze_rows": bronze_count,
        "silver_rows": silver_count,
        "quarantine_rows": reject_count,
        "duplicates_removed": duplicate_rows_removed,
        "accounted_for_rows": accounted_for,
        "difference": difference,
        "status": "PASS" if difference == 0 else "FAIL"
    }
]

display(
    spark.createDataFrame(result)
)

accounted_for_rows,bronze_rows,difference,duplicates_removed,entity,quarantine_rows,silver_rows,status
1200078,1200078,0,0,addresses,21678,1178400,PASS


Analyze reject reasons

In [0]:
(
    reject_df
    .groupBy("rejection_reason")
    .count()
    .orderBy(F.desc("count"))
    .show(truncate=False)
)

+----------------------------------------------------------------+-----+
|rejection_reason                                                |count|
+----------------------------------------------------------------+-----+
|CUSTOMER_NOT_FOUND                                              |19200|
|INVALID_EFFECTIVE_DATES                                         |2400 |
|MISSING_CUSTOMER_ID; CUSTOMER_NOT_FOUND; INVALID_EFFECTIVE_DATES|78   |
+----------------------------------------------------------------+-----+



Validate referential integrity in Silver

In [0]:
orphan_count = (
    silver_df.alias("a")
    .join(
        spark.table(
            "customer360_dev.silver.customers"
        ).select("customer_id").alias("c"),
        on="customer_id",
        how="left_anti"
    )
    .count()
)

print(
    "Orphan addresses in Silver:",
    orphan_count
)

Orphan addresses in Silver: 0


Measure primary-address conflicts

In [0]:
primary_conflict_count = (
    silver_df
    .filter(
        (F.col("is_primary") == True)
        & (
            F.col("effective_to").isNull()
        )
    )
    .groupBy(
        "customer_id",
        "address_type"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

print(
    "Primary-address conflict groups:",
    primary_conflict_count
)

Primary-address conflict groups: 0
